# ⚖️ RAG for Legal

## Why Legal Is a Great RAG Use Case

Lawyers and paralegals spend enormous time searching through:
- Case law and precedents
- Contracts and agreements
- Regulatory filings

RAG can surface the right clause, case, or regulation in seconds.

## What's Special About Legal Documents?

```
1. Precision matters enormously — one wrong clause = big problem
2. Source citation is mandatory — always show where it came from
3. Jurisdiction matters — case law varies by state/country
4. Dense language — lots of legal jargon to handle
```

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


In [2]:
# Simulated legal document chunks
# In production: load from contract PDFs, case law databases, etc.

legal_docs = [
    {"source": "NDA Agreement §3",          "text": "The Receiving Party shall maintain the Confidential Information in strict confidence and shall not disclose it to any third party without prior written consent."},
    {"source": "NDA Agreement §5",          "text": "This Agreement shall terminate two years from the Effective Date, unless earlier terminated by mutual written agreement."},
    {"source": "Employment Contract §7",     "text": "Employee agrees not to engage in any competing business activities during the term of employment and for 12 months following termination."},
    {"source": "Employment Contract §12",    "text": "All intellectual property created by Employee in the course of employment shall be the exclusive property of the Company."},
    {"source": "Terms of Service §4.2",      "text": "The Company reserves the right to terminate user access for violation of these terms, including but not limited to fraudulent activity."},
    {"source": "Privacy Policy §2",          "text": "We collect personal data including name, email, and IP address. Data is retained for no more than 36 months from last activity."},
    {"source": "Service Agreement §8",       "text": "Either party may terminate this agreement with 30 days written notice. Early termination by Client incurs a fee of 25% of remaining contract value."},
]

texts      = [d["text"] for d in legal_docs]
embeddings = model.encode(texts)
print(f"Indexed {len(legal_docs)} legal clauses")

Indexed 7 legal clauses


In [3]:
# Legal search — ALWAYS show the source

def legal_search(question, top_k=2):
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    print(f"Question: '{question}'")
    print("-" * 60)
    for rank, idx in enumerate(top_idx, 1):
        doc = legal_docs[idx]
        print(f"  {rank}. [{scores[idx]:.3f}] Source: {doc['source']}")
        print(f"     {doc['text']}")
    print()
    print("⚠️  Always verify with a qualified legal professional.")
    print()

legal_search("Can I share confidential information with partners?")
legal_search("What happens to IP I create at work?")
legal_search("How do I cancel a service contract early?")

Question: 'Can I share confidential information with partners?'
------------------------------------------------------------
  1. [0.542] Source: NDA Agreement §3
     The Receiving Party shall maintain the Confidential Information in strict confidence and shall not disclose it to any third party without prior written consent.
  2. [0.310] Source: Privacy Policy §2
     We collect personal data including name, email, and IP address. Data is retained for no more than 36 months from last activity.

⚠️  Always verify with a qualified legal professional.

Question: 'What happens to IP I create at work?'
------------------------------------------------------------
  1. [0.418] Source: Employment Contract §12
     All intellectual property created by Employee in the course of employment shall be the exclusive property of the Company.
  2. [0.267] Source: Terms of Service §4.2
     The Company reserves the right to terminate user access for violation of these terms, including but not limited 

In [4]:
# Legal-specific prompt — conservative, always cites sources, never gives legal advice

def legal_prompt(question, top_k=2):
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    context = "\n\n".join(
        f"[{legal_docs[i]['source']}]\n{legal_docs[i]['text']}"
        for i in top_idx
    )

    return f"""You are a legal document assistant. Your job is to find relevant clauses, not give legal advice.
Always cite the exact source. If you're unsure, say so.

Relevant clauses:
{context}

Question: {question}

Answer (cite sources, add disclaimer):"""

print(legal_prompt("How long does an NDA last?"))

You are a legal document assistant. Your job is to find relevant clauses, not give legal advice.
Always cite the exact source. If you're unsure, say so.

Relevant clauses:
[Privacy Policy §2]
We collect personal data including name, email, and IP address. Data is retained for no more than 36 months from last activity.

[NDA Agreement §5]
This Agreement shall terminate two years from the Effective Date, unless earlier terminated by mutual written agreement.

Question: How long does an NDA last?

Answer (cite sources, add disclaimer):


## Key Rules for Legal RAG

1. **Always cite the source** — section number, document name, date
2. **Never give legal advice** — present clauses, not interpretations
3. **Require high confidence** — set a strict relevance threshold
4. **Log everything** — audit trails matter in legal contexts
5. **Human in the loop** — always recommend consulting a lawyer